Part B: Train XGBoost on SageMaker
====================================================
Prereqs:
  - train.csv and test.csv already uploaded to your S3 bucket
    (SageMaker XGBoost CSV format: target column first, NO header)
  - Notebook instance IAM role has S3 + SageMaker permissions

This script:
  1. Points the built-in SageMaker XGBoost algorithm at your S3 train/test data
  2. Runs training with (near-)default hyperparameters -- only 'objective' and
     'num_round' are set, since XGBoost built-in requires objective to be
     specified explicitly for regression; everything else uses XGBoost's
     own internal defaults
  3. Does NOT deploy a real-time endpoint
  4. Evaluates the trained model by downloading the model artifact from S3
     and running predictions locally in the notebook with the xgboost library
  5. Prints RMSE, MAE, and R^2 on your test set
"""

In [1]:
import sys
!{sys.executable} -m pip install xgboost==1.7.6

  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 82.2 MB/s  0:00:02m0:00:0100:01
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]


In [2]:
# --- CELL 1: Setup ---
import boto3
import sagemaker
import pandas as pd
import numpy as np
import tarfile
import os

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()  # uses the notebook instance's IAM role

# ------------------------------------------------------------------
# EDIT THESE to match your setup
# ------------------------------------------------------------------
BUCKET = "car-data-processed-ak"          # <-- your S3 bucket (no s3:// prefix)
PREFIX = "preprocessed_data"            # <-- folder/prefix inside the bucket
TRAIN_KEY = f"{PREFIX}/train.csv"    # <-- path to train.csv inside the bucket
TEST_KEY = f"{PREFIX}/test.csv"      # <-- path to test.csv inside the bucket

s3_train_path = f"s3://{BUCKET}/{TRAIN_KEY}"
s3_test_path = f"s3://{BUCKET}/{TEST_KEY}"
s3_output_path = f"s3://{BUCKET}/output"

print(f"Region: {region}")
print(f"Train data: {s3_train_path}")
print(f"Test data:  {s3_test_path}")
print(f"Model output: {s3_output_path}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Region: eu-west-1
Train data: s3://car-data-processed-ak/preprocessed_data/train.csv
Test data:  s3://car-data-processed-ak/preprocessed_data/test.csv
Model output: s3://car-data-processed-ak/output


In [3]:
# --- CELL 2: Point the built-in XGBoost algorithm at S3 ---
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker import image_uris

xgboost_image = image_uris.retrieve("xgboost", region, version="1.7-1")

train_input = TrainingInput(s3_train_path, content_type="text/csv")
test_input = TrainingInput(s3_test_path, content_type="text/csv")

In [4]:
# --- CELL 3: Configure & launch the training job ---
# NOTE: train_instance_type here is the EPHEMERAL training compute --
# separate from your ml.t3.medium notebook instance. It spins up only for
# the duration of the job and shuts itself down automatically afterward.
xgb_estimator = Estimator(
    image_uri=xgboost_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",   # training compute, NOT the notebook instance
    output_path=s3_output_path,
    sagemaker_session=session,
)

# "Default hyperparameters": only objective + num_round are set explicitly
# (objective must be specified for the built-in algorithm to know this is
# regression, not classification). Everything else -- max_depth, eta, subsample,
# etc. -- is left at XGBoost's own internal defaults, i.e. untouched/untuned.
xgb_estimator.set_hyperparameters(
    objective="reg:squarederror",
    num_round=100,
)

# This blocks until training completes; the training instance is torn down
# automatically by SageMaker the moment the job finishes (whether it succeeds
# or fails) -- you are not billed for it beyond the training duration.
xgb_estimator.fit({"train": train_input, "validation": test_input})

print("Training complete.")
print("Model artifact location:", xgb_estimator.model_data)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-08-07-14-27-35-242


2026-08-07 14:27:36 Starting - Starting the training job...
2026-08-07 14:27:49 Starting - Preparing the instances for training...
2026-08-07 14:28:12 Downloading - Downloading input data...
2026-08-07 14:29:03 Downloading - Downloading the training image......
2026-08-07 14:30:08 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-08-07 14:30:10.276 ip-10-0-224-240.eu-west-1.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-08-07 14:30:10.350 ip-10-0-224-240.eu-west-1.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-08-07:14:30:10:INFO] Imported framework sagem

In [5]:
# --- CELL 4: Evaluate on the test set WITHOUT deploying an endpoint ---
# Download the trained model artifact from S3 and run predictions locally
# in this notebook using the xgboost library directly. This avoids creating
# any additional SageMaker-hosted resource (no real-time endpoint, no
# persistent hosting infrastructure).
import urllib.request
from urllib.parse import urlparse

model_data_url = xgb_estimator.model_data
parsed = urlparse(model_data_url)
bucket_name = parsed.netloc
model_key = parsed.path.lstrip("/")

s3_client = boto3.client("s3")
local_model_tar = "model.tar.gz"
s3_client.download_file(bucket_name, model_key, local_model_tar)

with tarfile.open(local_model_tar) as tar:
    tar.extractall(path="model_dir")

print("Extracted model files:", os.listdir("model_dir"))

Extracted model files: ['xgboost-model', '.ipynb_checkpoints']


In [6]:
# --- CELL 5: Load the model and predict on the test set ---
import xgboost as xgb

# The built-in algorithm saves the model as a raw XGBoost Booster file
booster = xgb.Booster()
booster.load_model("model_dir/xgboost-model")

# Download test.csv locally to run predictions
s3_client.download_file(BUCKET, TEST_KEY, "test_local.csv")

# Recall: test.csv has NO header, target (price) is the FIRST column
test_df = pd.read_csv("test_local.csv", header=None)
y_true = test_df.iloc[:, 0].values
X_test = test_df.iloc[:, 1:].values

dtest = xgb.DMatrix(X_test)
y_pred = booster.predict(dtest)

In [7]:
import sys
!{sys.executable} -m pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [8]:
# --- CELL 6: Metrics ---

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"R^2:  {r2:.4f}")

RMSE: 8,224.95
MAE:  5,523.28
R^2:  0.8401


In [9]:
# --- CELL 7: Feature importance (bonus -- useful for your earlier
# transmission_speeds question; check where it ranks) ---
importance = booster.get_score(importance_type="gain")
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)
print("\nTop features by gain:")
for feat, score in importance_sorted[:15]:
    print(f"  {feat}: {score:.2f}")

print("\n" + "=" * 60)
print("Evaluation complete. No real-time endpoint was created.")
print("Go to the SageMaker console NOW and stop this notebook instance.")
print("=" * 60)


Top features by gain:
  f49: 1530539520.00
  f2: 1078041088.00
  f4: 985082944.00
  f1: 968408704.00
  f63: 792135680.00
  f0: 595427904.00
  f67: 560846976.00
  f58: 507920320.00
  f16: 483849408.00
  f35: 464002880.00
  f3: 441600928.00
  f29: 407303296.00
  f50: 388314400.00
  f41: 374023328.00
  f36: 366727168.00

Evaluation complete. No real-time endpoint was created.
Go to the SageMaker console NOW and stop this notebook instance.
